# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prajwal-Rd-bond/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring** (Lane 2 in `docs/ml-intern-dataset-and-lane-guide.md`).

I'm picking this lane instead of freestyling because it's the one the shipped starter pipeline
(`scripts/01`-`05`) already demonstrates end to end, so I can check every number I claim against a
real, reproducible reference (`outputs/model_report.md`). The underlying problem is also concrete:
FlyRank manages many pages per client, and a human reviewer can only look at a handful per week.
Deciding *which* pages get that limited attention first is a real, standing decision, not a
hypothetical one -- and it's a decision a ranked, evidence-backed queue can genuinely improve.


In [1]:
import os, sys
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Pages:", df.shape[0], "| Clients:", df["client_id"].nunique())
print("Median pages per client:", int(df.groupby("client_id").size().median()))
print("Max pages for one client:", int(df.groupby("client_id").size().max()))


Pages: 30000 | Clients: 32
Median pages per client: 567
Max pages for one client: 7008


## 2. The question: decision, action, cost of a wrong call

**Decision:** which pages a content editor should open and review first this week, out of
everything a client owns.

**Who acts:** a FlyRank content editor / SEO strategist with a fixed weekly review capacity --
not everyone, not everything.

**Cost of a wrong call** is two-sided:
- *False positive* (flagged, not actually a problem): an editor spends real time reviewing a page
  that didn't need it -- time taken away from a page that did.
- *False negative* (a real decliner never surfaces): the page keeps losing search visibility
  silently until someone notices by accident, weeks later, when the fix is harder and costlier.

Because review capacity is small relative to the total page count (checked below), the *order* at
the top of the queue matters far more than getting every single page right -- exactly what a
precision-at-K question rewards, not plain accuracy.


In [2]:
capacity_per_week = 50  # a realistic per-client weekly review capacity for one editor
median_pages = int(df.groupby("client_id").size().median())
print(f"Median pages per client: {median_pages}")
print(
    f"At {capacity_per_week} reviews/week, a median client's backlog alone takes "
    f"~{median_pages / capacity_per_week:.1f} weeks to review once end to end -- "
    "prioritization order is not optional, it's the whole game."
)


Median pages per client: 567
At 50 reviews/week, a median client's backlog alone takes ~11.3 weeks to review once end to end -- prioritization order is not optional, it's the whole game.


## 3. Quick look at the data (2-3 real numbers)

Three real numbers from the starter slice that make this lane worth the next 7 weeks:


In [3]:
declining_rate = df["trend_direction"].str.lower().eq("down").mean()
corr = df["search_volume"].corr(df["impressions_90d"])
declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).mean()

print(f"1) {declining_rate:.1%} of pages are currently in a 'down' trend "
      f"(impressions fell >20%, last 30d vs prior 30d).")
print(f"2) search_volume barely predicts real traffic: "
      f"corr(search_volume, impressions_90d) = {corr:.3f} -- so 'high demand keyword' alone is a weak signal.")
print(f"3) {declining_with_demand:.1%} of pages are declining WHILE still getting real search demand "
      f"(100+ impressions/90d) -- these are exactly the pages worth reviewing first, since a decline")
print(f"   with no demand behind it barely matters, but this population is large enough that prioritizing")
print(f"   *within* it is the real problem this lane exists to solve.")


1) 54.2% of pages are currently in a 'down' trend (impressions fell >20%, last 30d vs prior 30d).
2) search_volume barely predicts real traffic: corr(search_volume, impressions_90d) = 0.001 -- so 'high demand keyword' alone is a weak signal.
3) 43.8% of pages are declining WHILE still getting real search demand (100+ impressions/90d) -- these are exactly the pages worth reviewing first, since a decline
   with no demand behind it barely matters, but this population is large enough that prioritizing
   *within* it is the real problem this lane exists to solve.


## 4. Careful words: what I can and can't claim

**What this work CAN say:** observed rates and patterns in this anonymized slice (e.g. "X% of
pages show a >20% impression drop over the last 30 days"); directional relationships ("stale and
visible pages are disproportionately declining"); decision-support rankings ("these are the top
candidates to review first, and here is why, in plain reason codes").

**What it NEVER will say:** that a refresh *caused* a recovery (no experiment ran -- see
`docs/ml-intern-dataset-and-lane-guide.md` section 7); that any result generalizes to how Google's
live ranking algorithm works; that a flagged page is guaranteed to improve if reviewed. Every
output stays a ranked recommendation for a human reviewer, never a verdict.


In [4]:
# Sanity check: is my label observed ground truth, or a rule someone hand-picked?
print("Label source: trend_direction, itself derived from a fixed +/-20% threshold on trend_pct.")
print("That makes it a DEFINED PROXY, not a fully independent future outcome -- worth restating")
print("honestly in every write-up from here on (see Week-2 notebook for more on this).")
print(f"\nBase rate to compare every future metric against: {declining_rate:.3f}")


Label source: trend_direction, itself derived from a fixed +/-20% threshold on trend_pct.
That makes it a DEFINED PROXY, not a fully independent future outcome -- worth restating
honestly in every write-up from here on (see Week-2 notebook for more on this).

Base rate to compare every future metric against: 0.542


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.